In [ ]:
import pandas as pd
import utilities.weather_api as weather

df = pd.read_csv("../datasets/forest_features_20240702.csv")
df

Adding a column with the average temperature for the week. This uses past 
forecasts (not the actual recorded temperatures), so we are not incorporating 
future 
information into our prediction model. 

In [ ]:
df["avg_temp"] = list(
    map(
        weather.get_avg_weekly_forecast_from_loc_code,
        [str(code) for code in df["loc_code"]],
        df["date"],
    )
)

In [ ]:
df

In [ ]:
df.to_csv("../datasets/forest_features_20240708.csv", index=False)

We will train 3 different models. Then combine them into an ensemble voting 
model. 

In [ ]:
"""Prepare the data."""

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

df.dropna(inplace=True)

features = [
    "1_week_roc",
    "week_of_year",
    "moving_avg_WIS",
    "lagged_1wk_WIS",
    "holiday_last_week",
    "avg_temp",
]
X = df[features]
y = df["is_prediction_over"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
"""Train and evaluate the RF model."""

from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

# Evaluate the model
y_pred_rf = rf_model.predict(X_test)
print(classification_report(y_test, y_pred_rf))

In [ ]:
import matplotlib.pyplot as plt

# Feature Importances
importances = rf_model.feature_importances_
feature_names = X_train.columns

# Plot Feature Importances
plt.figure(figsize=(10, 6), dpi=200)
plt.barh(feature_names, importances, color="skyblue")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.title("Feature Importance")
plt.show()

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

# Initialize models
rf_model = RandomForestClassifier(random_state=42)
lr_model = LogisticRegression(random_state=42)
svc_model = SVC(probability=True, random_state=42)

# Train models
rf_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)
svc_model.fit(X_train, y_train)

# Voting Classifier
voting_clf_hard = VotingClassifier(
    estimators=[("rf", rf_model), ("lr", lr_model), ("svc", svc_model)], voting="hard"
)
voting_clf_soft = VotingClassifier(
    estimators=[("rf", rf_model), ("lr", lr_model), ("svc", svc_model)], voting="soft"
)

# Train Voting Classifier
voting_clf_hard.fit(X_train, y_train)
voting_clf_soft.fit(X_train, y_train)

# Evaluate models
print("Random Forest Classifier")
print(classification_report(y_test, rf_model.predict(X_test)))

print("Logistic Regression Classifier")
print(classification_report(y_test, lr_model.predict(X_test)))

print("SVC Classifier")
print(classification_report(y_test, svc_model.predict(X_test)))

print("Hard Voting Classifier")
print(classification_report(y_test, voting_clf_hard.predict(X_test)))

print("Soft Voting Classifier")
print(classification_report(y_test, voting_clf_soft.predict(X_test)))

Hmm. This didn't work as well as I wanted it to. We should implement 
different, uncorrelated features for each model, so we don't have the 
correlated error that is liekly present here.